# 05b — Temporal Persistence Improvement Experiments

The same kind of hypothesis-testing `03b_model_improvements.ipynb` did for the panel CatBoost model, applied to `05_temporal_persistence_check.ipynb`'s best result — SARIMAX with `Alcohol use disorders` as a single exogenous feature (Val R² ≈ 0.74).

The four hypotheses do not translate literally: SARIMAX + 1 exog has a single exogenous feature, not a block of correlated ones, so "multicollinearity" and "feature trimming" have no direct equivalent here. Each hypothesis below is adapted to ask the closest analogous question for a per-country time-series model instead of dropped silently:

1. **Sensitivity to *which* exogenous feature was chosen** — the closest analogue to a multicollinearity check: is `Alcohol use disorders` doing something special, or would another determinant work about as well (in which case the specific choice would be closer to arbitrary)?
2. **Is the Test/Val gap a COVID-period effect?** — directly transferable, same test as `03b`.
3. **Does adding more exogenous features help?** — the mirror image of `03b`'s feature-trimming test: `03b` removed low-importance features from a many-feature panel model; there is only one feature to remove here, so this asks the opposite direction — does adding a second and third feature help or hurt.
4. **Is the ARIMA order (1,1,0) actually the best choice?** — the direct analogue of `03b`'s extended-grid check, testing alternative (p,d,q) orders instead of an extended `l2_leaf_reg` range.

Unlike `03b`, where none of the four hypotheses that touched the model's configuration held up, **one of these four does** — see Hypothesis 4.

In [3]:
import sys

sys.path.append("..")

import warnings
import pandas as pd

from src import TARGET, temporal_split, metrics_by_period
from src.timeseries_models import (
    train_evaluate_sarimax,
    fit_sarimax_models,
    forecast_sarimax,
)

pd.set_option("display.width", 1000)

df_development = pd.read_parquet("../data/processed/df_development.parquet")
df_train_B, df_test_B, df_val_B, *_ = temporal_split(df_development)

BASELINE_EXOG = "Alcohol use disorders"
BASELINE_ORDER = (1, 1, 0)

eval_baseline, _ = train_evaluate_sarimax(
    df_train_B,
    df_test_B,
    df_val_B,
    TARGET,
    exog_features=[BASELINE_EXOG],
    order=BASELINE_ORDER,
)
baseline_test = [e for e in eval_baseline if e["split"] == "Test"][0]
baseline_val = [e for e in eval_baseline if e["split"] == "Val"][0]
print(
    f"Baseline (order={BASELINE_ORDER}, exog={BASELINE_EXOG}) — Test R²: {baseline_test['r2']:.4f} | Val R²: {baseline_val['r2']:.4f}"
)

Baseline (order=(1, 1, 0), exog=Alcohol use disorders) — Test R²: 0.9358 | Val R²: 0.7439


## Hypothesis 1 — Sensitivity to which exogenous feature is chosen

Refits SARIMAX with the same order, swapping in one alternative single exogenous feature at a time: two other mental-health indicators (`ADHD`, `Bipolar disorder`, both plausible from the CatBoost SHAP ranking in Section 3.6.3) and two socioeconomic ones (`GDP per capita`, `Unemployment rate (%)`).

In [4]:
candidate_features = [
    "Alcohol use disorders",  # baseline
    "Attention-deficit/hyperactivity disorder",
    "Bipolar disorder",
    "GDP per capita",
    "Unemployment rate (%)",
]

rows = []
for feat in candidate_features:
    eval_r, _ = train_evaluate_sarimax(
        df_train_B,
        df_test_B,
        df_val_B,
        TARGET,
        exog_features=[feat],
        order=BASELINE_ORDER,
    )
    test_e = [e for e in eval_r if e["split"] == "Test"][0]
    val_e = [e for e in eval_r if e["split"] == "Val"][0]
    rows.append(
        {"Exogenous feature": feat, "Test R²": test_e["r2"], "Val R²": val_e["r2"]}
    )

h1_results = pd.DataFrame(rows).sort_values("Val R²", ascending=False)
h1_results

,Exogenous feature,Test R²,Val R²
0,Alcohol use disorders,0.9358,0.7439
3,GDP per capita,0.8832,0.7199
1,Attention-deficit/hyperactivity disorder,0.9209,0.6788
4,Unemployment rate (%),0.9226,0.6641
2,Bipolar disorder,0.8917,0.4075


**Result: not fully confirmed, but not fully rejected either.** `Alcohol use disorders` is genuinely the best of the five candidates (Val R² 0.744), not an arbitrary pick that happened to be reported — but it isn't uniquely so: `GDP per capita` comes close (0.720), while `Bipolar disorder` is clearly worse (0.408). The choice of exogenous feature matters, and the one used throughout Sections 3.8–3.9 is a reasonable, close-to-best choice rather than either a lucky coincidence or the only option that would have worked.

## Hypothesis 2 — Is the Test/Val gap a COVID-period effect?

Same test as `03b`'s Hypothesis 2: split the Validation years into Pre-COVID (2018–2019) and COVID (2020–2021) and compare R² within each, using `src/metrics.py::metrics_by_period`.

In [5]:
df_train_B_sorted = df_train_B.sort_values(["Year", "Code"]).reset_index(drop=True)
models = fit_sarimax_models(
    df_train_B_sorted, TARGET, exog_features=[BASELINE_EXOG], order=BASELINE_ORDER
)

future_block = pd.concat([df_test_B, df_val_B]).sort_values(["Code", "Year"])
rows = []
for code, group in df_val_B.groupby("Code"):
    if code not in models:
        continue
    country_future = future_block[future_block["Code"] == code]
    forecast = forecast_sarimax(
        models, code, country_future, exog_features=[BASELINE_EXOG]
    )
    for _, row in group.iterrows():
        rows.append(
            {
                "Year": row["Year"],
                "actual": row[TARGET],
                "pred": forecast.loc[row["Year"]],
            }
        )

val_predictions = pd.DataFrame(rows)
periods = {"Pre-COVID (2018-2019)": [2018, 2019], "COVID (2020-2021)": [2020, 2021]}
h2_results = metrics_by_period(
    val_predictions["actual"].to_numpy(),
    val_predictions["pred"].to_numpy(),
    val_predictions["Year"].to_numpy(),
    periods,
)
h2_results

,Period,N,RMSE,MAE,R²
0,Pre-COVID (2018-2019),54,2.1557,1.7093,0.8021
1,COVID (2020-2021),54,2.5876,1.9267,0.6775


**Result: confirmed, same direction as `03b`, smaller magnitude.** R² drops from 0.80 (Pre-COVID) to 0.68 (COVID) — a 0.12 drop, versus CatBoost's 0.72 → 0.47 (a 0.25 drop). The COVID-period effect is real for the temporal model too, but noticeably softer: a per-country forecast already "expects" some growing uncertainty the further it extrapolates from training, so a structural shock late in that horizon compounds a softer, already-present trend rather than hitting a comparatively flat-response panel model without warning.

## Hypothesis 3 — Does adding more exogenous features help?

The mirror image of `03b`'s feature-trimming test. `03b` removed low-importance features from CatBoost's many-feature set and found it made things slightly worse; there is only one feature to remove from SARIMAX + 1 exog, so this asks the reverse question — adding a second and third feature, in SHAP-importance order from Section 3.6.3.

In [6]:
feature_sets = {
    "1 feature (baseline)": ["Alcohol use disorders"],
    "2 features": ["Alcohol use disorders", "GDP per capita"],
    "3 features": ["Alcohol use disorders", "GDP per capita", "Unemployment rate (%)"],
}

rows = []
for label, feats in feature_sets.items():
    eval_r, _ = train_evaluate_sarimax(
        df_train_B,
        df_test_B,
        df_val_B,
        TARGET,
        exog_features=feats,
        order=BASELINE_ORDER,
    )
    test_e = [e for e in eval_r if e["split"] == "Test"][0]
    val_e = [e for e in eval_r if e["split"] == "Val"][0]
    rows.append({"Feature set": label, "Test R²": test_e["r2"], "Val R²": val_e["r2"]})

h3_results = pd.DataFrame(rows)
h3_results

,Feature set,Test R²,Val R²
0,1 feature (baseline),0.9358,0.7439
1,2 features,0.9058,0.7181
2,3 features,0.8887,0.6887


**Result: not confirmed — more features make it worse, monotonically.** Val R² drops from 0.744 (1 feature) to 0.718 (2) to 0.689 (3), mirroring `03b`'s finding for CatBoost from the opposite direction: both point to "the currently-used feature set is already close to the right size for this amount of data," not to "more determinants would help if only they were included." With ~15 training points per country, this is consistent with the overfitting risk flagged in Section 3.8's exogenous-variables discussion — each additional regressor is one more parameter competing for the same small sample.

## Hypothesis 4 — Is ARIMA order (1,1,0) actually the best choice?

The direct analogue of `03b`'s extended-`l2_leaf_reg`-range check: does a different (p,d,q) order improve on the (1,1,0) used throughout Sections 3.8–3.9?

In [7]:
orders_to_test = [(1, 1, 0), (2, 1, 0), (1, 1, 1), (0, 1, 1), (2, 1, 1)]

rows = []
for order in orders_to_test:
    eval_r, _ = train_evaluate_sarimax(
        df_train_B,
        df_test_B,
        df_val_B,
        TARGET,
        exog_features=[BASELINE_EXOG],
        order=order,
    )
    test_e = [e for e in eval_r if e["split"] == "Test"][0]
    val_e = [e for e in eval_r if e["split"] == "Val"][0]
    rows.append(
        {"Order (p,d,q)": str(order), "Test R²": test_e["r2"], "Val R²": val_e["r2"]}
    )

h4_results = pd.DataFrame(rows).sort_values("Val R²", ascending=False)
h4_results

,"Order (p,d,q)",Test R²,Val R²
2,"(1, 1, 1)",0.9401,0.7654
0,"(1, 1, 0)",0.9358,0.7439
1,"(2, 1, 0)",0.9305,0.7404
4,"(2, 1, 1)",0.9303,0.7386
3,"(0, 1, 1)",0.9358,0.7362


**Result: confirmed — unlike every other hypothesis tested in this notebook or in `03b`, this one actually changed the recommended configuration.** Order (1,1,1) — adding a first-order moving-average term — beats the (1,1,0) baseline on both Test (0.940 vs 0.936) and Val (0.765 vs 0.744) R². The improvement is modest but consistent across both splits, which is a more convincing signal than an improvement on only one of the two.

**Why the MA(1) term helps, not just that it does.** (1,1,0) models each country's year-over-year suicide rate as depending only on its own previous *level* (through the AR(1) term) after differencing. Adding an MA(1) term lets the model also account for last period's *forecast error* — useful here because a country's socioeconomic/mental-health situation rarely shifts cleanly in one step; a one-off shock (a data revision, a genuinely unusual year) tends to leave a short-lived echo in the following year's error that a pure AR term cannot capture but an MA term can. With exactly one covariate and ≈15 points per country, this is a single extra parameter, not a bigger search space — cheap enough not to reopen the overfitting risk flagged for the exogenous-feature grid in Section 3.8.

**Adopted.** `order=(1,1,1)` is the default in `src/timeseries_models.py::train_evaluate_sarimax`/`fit_sarimax_models`, so `05_temporal_persistence_check.py`/`.ipynb` and Sections 3.8–3.9 pick it up without needing to pass `order` explicitly; `06_visualize_predictions.py`/`.ipynb` pass `order` directly and were updated to `(1,1,1)` to match. The comparison above — testing the *old* (1,1,0) against four alternatives, on data generated before the change — is kept as the record of the experiment that produced this decision, not as an open question.

## Summary

| Hypothesis | Result |
|---|---|
| 1. Sensitivity to exogenous feature choice | Partially confirmed — `Alcohol use disorders` is genuinely near-best (0.744), not uniquely so (`GDP per capita` reaches 0.720), but a poor choice (`Bipolar disorder`, 0.408) is clearly worse |
| 2. Test/Val gap is a COVID-period effect | Confirmed — Val R² 0.80 (Pre-COVID) → 0.68 (COVID), same direction as `03b`, about half the magnitude |
| 3. Adding more exogenous features helps | Not confirmed — Val R² falls monotonically as features are added (0.744 → 0.718 → 0.689) |
| 4. Order (1,1,0) is the best choice | **Not confirmed as optimal** — order (1,1,1) beats it on both Test and Val R² |

Three of four hypotheses replicate `03b`'s broader pattern (the model's current configuration is already close to sensible, most proposed changes make it slightly worse). The fourth is the exception worth carrying forward: this notebook found a real, if modest, improvement that `03b` did not find for CatBoost's analogous check.